## InternVideo2 — Video + Text Shared Semantic Space

Focused experiment: shared semantic space between **video** and **text** using
InternVideo2 Stage2 as the sole encoder.

### Architecture
```
MP4 ──► segment into clips ──► InternVideo2 video encoder ──► Qdrant  iv2_only_v1
                │
                └──► ffmpeg audio ──► Whisper ──► transcript (LLM context only)

text query ──► BERT-large (IV2 text encoder) ──► similarity search ──► clips
                                                          │
                                                 attach transcript ──► LLM
```

### API notes
The HuggingFace checkpoint `OpenGVLab/InternVideo2-Stage2_1B-224p-f4` ships a
custom `modeling_internvideo2.py` loaded via `trust_remote_code=True`.
The actual methods are:
- **`model.get_vid_feat(tensor)`** — video encoder, input shape `(B, C, T, H, W)`
- **`model.get_txt_feat(input_ids, attention_mask, token_type_ids)`** — BERT-large text encoder
- Helper imports: `vid2tensor`, `_frame_from_video`, `retrieve_text` from `modeling_internvideo2`

> **Verified checkpoints on HuggingFace (Stage2, retrieval-optimised):**
> - `OpenGVLab/InternVideo2-Stage2_1B-224p-f4` — 1B, 4 frames, **default**
> - `OpenGVLab/InternVideo2-Stage2_1B-224p-f8` — 1B, 8 frames, better temporal resolution
> - `OpenGVLab/InternVideo2-Stage2_6B-224p-f4` — 6B, ~12 GB VRAM in fp16
> - `OpenGVLab/InternVideo2-CLIP-1B-224p-f8` — CLIP-style contrastive head variant


### 1. Configuration

In [60]:
import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv("setup.env", override=True)

# ── Input ─────────────────────────────────────────────────────────────────────
VIDEO_PATH = os.getenv("VIDEO_PATH", "./content/video.mp4")

# ── InternVideo2 Stage2 ───────────────────────────────────────────────────────
# Only Stage2 checkpoints expose shared video-text embedding space.
# "OpenGVLab/InternVideo2-Stage2_1B-224p-f4"   — 1B, 4 frames (default)
# "OpenGVLab/InternVideo2-Stage2_1B-224p-f8"   — 1B, 8 frames
# "OpenGVLab/InternVideo2-Stage2_6B-224p-f4"   — 6B (~12 GB VRAM fp16)
# "OpenGVLab/InternVideo2-CLIP-1B-224p-f8"     — CLIP-style head
IV2_MODEL       = os.getenv("IV2_MODEL", "OpenGVLab/InternVideo2-Stage2_1B-224p-f4")
IV2_NUM_FRAMES  = int(os.getenv("IV2_NUM_FRAMES",  "8"))   # must match checkpoint suffix -f4 / -f8
IV2_SEGMENT_SECS= int(os.getenv("IV2_SEGMENT_SECS","8"))  # seconds per video clip
IV2_OVERLAP_SECS= int(os.getenv("IV2_OVERLAP_SECS", "2"))  # overlap between clips
IV2_BATCH_SIZE  = int(os.getenv("IV2_BATCH_SIZE",   "4"))  # clips per forward pass

# ── BERT tokenizer (text encoder of InternVideo2 Stage2) ─────────────────────
# Stage2 uses bert-large-uncased as text encoder — NOT the IV2 tokenizer.
BERT_MODEL      = "bert-large-uncased"
TEXT_MAX_LENGTH = int(os.getenv("TEXT_MAX_LENGTH", "77"))

# ── Whisper (transcript for LLM context only — not used for retrieval) ────────
WHISPER_MODEL    = os.getenv("WHISPER_MODEL",    "openai/whisper-large-v3")
WHISPER_LANGUAGE = os.getenv("WHISPER_LANGUAGE", "en")
SAMPLE_RATE      = 16000

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVER_K    = int(os.getenv("RETRIEVER_K",    "8"))
RERANKER_TOP_N = int(os.getenv("RERANKER_TOP_N", "4"))
RERANKER_MODEL = os.getenv("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
ENABLE_RERANKING = os.getenv("ENABLE_RERANKING", "true").lower() == "true"

# ── Generation ────────────────────────────────────────────────────────────────
GENERATION_MODEL   = os.getenv("GENERATION_MODEL",   "Qwen/Qwen2.5-3B-Instruct")
GENERATION_BACKEND = os.getenv("GENERATION_BACKEND", "hf")  # "hf" | "ollama"
GENERATION_MAX_NEW_TOKENS = int(os.getenv("GENERATION_MAX_NEW_TOKENS", "512"))
HF_DEVICE_MAP  = os.getenv("HF_DEVICE_MAP",  "auto")
HF_TORCH_DTYPE = os.getenv("HF_TORCH_DTYPE", "auto")

# ── Qdrant ────────────────────────────────────────────────────────────────────
QDRANT_URL       = os.getenv("QDRANT_URL",     "http://localhost:6333")
QDRANT_API_KEY   = os.getenv("QDRANT_API_KEY", "")
QDRANT_IV2_ONLY  = os.getenv("QDRANT_IV2_ONLY", "iv2_only_v1")
RESET_COLLECTION = os.getenv("RESET_COLLECTION", "true").lower() == "false"

# ── Persistence + cross-notebook results ──────────────────────────────────────
PERSIST_DIR   = os.getenv("PERSIST_DIR",   "./cache/iv2_only/")
RESULTS_AUDIO = os.getenv("RESULTS_AUDIO", "./cache/audio/audio_eval_results.json")
RESULTS_FUSION= os.getenv("RESULTS_FUSION","./cache/multimodal/multimodal_eval_results.json")
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs("./content/", exist_ok=True)

print("Configuration loaded.")
print(f"  Video      : {VIDEO_PATH}")
print(f"  IV2 model  : {IV2_MODEL}")
print(f"  Frames/clip: {IV2_NUM_FRAMES}  | seg={IV2_SEGMENT_SECS}s | overlap={IV2_OVERLAP_SECS}s")
print(f"  BERT       : {BERT_MODEL}")
print(f"  Whisper    : {WHISPER_MODEL}")
print(f"  Generator  : {GENERATION_MODEL} ({GENERATION_BACKEND})")
print(f"  Qdrant     : {QDRANT_URL}  collection={QDRANT_IV2_ONLY}")


Configuration loaded.
  Video      : ./content/video.mp4
  IV2 model  : OpenGVLab/InternVideo2-Stage2_1B-224p-f4
  Frames/clip: 8  | seg=8s | overlap=2s
  BERT       : bert-large-uncased
  Whisper    : openai/whisper-large-v3
  Generator  : mistral-nemo:latest (ollama)
  Qdrant     : http://localhost:6333  collection=iv2_only_v1


### 2. Audio Extraction + Whisper Transcription

Whisper is used **only** for LLM context — not for retrieval.
Results are cached to avoid re-running on long videos.


In [61]:
import torch, numpy as np, json, hashlib, librosa
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

def _file_hash(path: str, n: int = 12) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""): h.update(chunk)
    return h.hexdigest()[:n]

# ── Extract audio track ───────────────────────────────────────────────────────
audio_track = Path(PERSIST_DIR) / f"audio_{_file_hash(VIDEO_PATH)}.wav"
if not audio_track.exists():
    print("Extracting audio track ...")
    rc = os.system(f'ffmpeg -y -i "{VIDEO_PATH}" -ar {SAMPLE_RATE} -ac 1 '
                   f'"{audio_track}" -loglevel error')
    if rc != 0 or not audio_track.exists():
        raise RuntimeError("ffmpeg failed. Verify ffmpeg is installed and VIDEO_PATH is valid.")
    print(f"  ✓ Saved: {audio_track.name}")
else:
    print(f"✓ Audio cached: {audio_track.name}")

# ── Whisper transcription ─────────────────────────────────────────────────────
transcript_cache = Path(PERSIST_DIR) / f"transcript_{_file_hash(VIDEO_PATH)}_{WHISPER_MODEL.split('/')[-1]}.json"

if transcript_cache.exists():
    with open(transcript_cache) as f:
        transcript_data = json.load(f)
    print(f"✓ Transcript cached: {transcript_cache.name} ({len(transcript_data['chunks'])} segments)")
else:
    print(f"Loading Whisper ({WHISPER_MODEL}) ...")
    _w_device = "cuda" if torch.cuda.is_available() else "cpu"
    _w_dtype  = torch.float16 if _w_device == "cuda" else torch.float32
    _w_proc   = AutoProcessor.from_pretrained(WHISPER_MODEL)
    _w_model  = AutoModelForSpeechSeq2Seq.from_pretrained(
        WHISPER_MODEL, torch_dtype=_w_dtype, low_cpu_mem_usage=True).to(_w_device)
    _w_pipe   = hf_pipeline(
        "automatic-speech-recognition",
        model=_w_model, tokenizer=_w_proc.tokenizer,
        feature_extractor=_w_proc.feature_extractor,
        torch_dtype=_w_dtype, device=_w_device,
        return_timestamps=True, chunk_length_s=30, batch_size=16,
        generate_kwargs={"language": WHISPER_LANGUAGE} if WHISPER_LANGUAGE else {},
    )
    waveform, _ = librosa.load(str(audio_track), sr=SAMPLE_RATE, mono=True)
    print(f"  Transcribing {len(waveform)/SAMPLE_RATE:.0f}s ...")
    result = _w_pipe(waveform.copy(), return_timestamps=True)
    transcript_data = {
        "text":   result["text"],
        "chunks": [{"text": c["text"], "timestamp": list(c["timestamp"])}
                   for c in result.get("chunks", [])],
    }
    with open(transcript_cache, "w") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    del _w_model, _w_pipe, _w_proc
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  ✓ Transcript saved ({len(transcript_data['chunks'])} segments).")

video_transcript_segs = transcript_data["chunks"]
print(f"\nFirst 400 chars: {transcript_data['text'][:400]}")


✓ Audio cached: audio_467e468b2f83.wav
✓ Transcript cached: transcript_467e468b2f83_whisper-large-v3.json (50 segments)

First 400 chars:  How did a single paper, attention is all you need, reshape the entire AI landscape? In this video, we will unpack the transformer architecture. We will see how it works, what makes it so powerful, and why it replaced almost every older neural network design. Before diving in, let's take a quick step back. The goal of machine learning is to learn a mapping from inputs to outputs for example in pre


### 3. InternVideo2 Stage2 — Model Loading

**Important API notes** verified against the official HuggingFace model cards:

- `AutoModel.from_pretrained(..., trust_remote_code=True)` downloads the custom
  `modeling_internvideo2.py` from HF and registers it. This file exposes the helper
  functions `vid2tensor`, `_frame_from_video`, `retrieve_text`.
- **Video encoder**: `model.get_vid_feat(video_tensor)` — input shape `(B, C, T, H, W)`.
- **Text encoder**: `model.get_txt_feat(input_ids, attention_mask, token_type_ids)` —
  the text encoder is **BERT-large** (`bert-large-uncased`), loaded separately.
- The embedding dim for both encoders is **1024** (Stage2 1B checkpoint).


In [62]:
# !git clone https://github.com/OpenGVLab/InternVideo.git

In [63]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, CLIPTokenizerFast

IV2_MODEL = "OpenGVLab/InternVideo2_CLIP_S"
IV2_TOKENIZER_MODEL = "openai/clip-vit-base-patch16"

IV2_AVAILABLE = False
IV2_DIM       = 512
iv2_model     = None
iv2_tokenizer = None

print(f"Loading InternVideo2: {IV2_MODEL} ...")

_iv2_device = "cuda" if torch.cuda.is_available() else "cpu"
_iv2_dtype  = torch.float32

try:
    iv2_model = AutoModel.from_pretrained(
        IV2_MODEL,
        torch_dtype=_iv2_dtype,
        trust_remote_code=True,
    ).to(_iv2_device).eval()

    # InternVideo2_CLIP_S has CLIP-like text config, but does not expose
    # a tokenizer compatible with AutoTokenizer.
    iv2_tokenizer = CLIPTokenizerFast.from_pretrained(IV2_TOKENIZER_MODEL)

    # ── Verify embedding dim ─────────────────────────────────────────────────
    _probe = iv2_tokenizer(
        ["probe"],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(_iv2_device)

    def _iv2_encode_text(tokenized):
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask", None)
        token_type_ids = tokenized.get("token_type_ids", None)

        if hasattr(iv2_model, "get_text_features"):
            return iv2_model.get_text_features(**tokenized)

        if hasattr(iv2_model, "get_txt_feat"):
            # Try the most common InternVideo2 signatures.
            try:
                if token_type_ids is not None:
                    return iv2_model.get_txt_feat(
                        input_ids,
                        attention_mask,
                        token_type_ids,
                    )
                return iv2_model.get_txt_feat(
                    input_ids,
                    attention_mask,
                )
            except TypeError:
                try:
                    return iv2_model.get_txt_feat(input_ids)
                except TypeError:
                    return iv2_model.get_txt_feat(
                        text=input_ids,
                        attention_mask=attention_mask,
                    )

        if hasattr(iv2_model, "encode_text"):
            try:
                return iv2_model.encode_text(input_ids)
            except TypeError:
                return iv2_model.encode_text(
                    input_ids,
                    attention_mask=attention_mask,
                )

        raise AttributeError(
            "No known text embedding method found. "
            "Try: [m for m in dir(iv2_model) "
            "if 'text' in m.lower() or 'txt' in m.lower() "
            "or 'encode' in m.lower() or 'feat' in m.lower()]"
        )

    with torch.no_grad():
        _txt_feat = _iv2_encode_text(_probe)
        _txt_feat = F.normalize(_txt_feat.float(), dim=-1)

    IV2_DIM = _txt_feat.shape[-1]

    n_params = sum(p.numel() for p in iv2_model.parameters()) / 1e6
    IV2_AVAILABLE = True

    print(
        f"  ✓ InternVideo2 ready | "
        f"device={_iv2_device} | "
        f"dim={IV2_DIM} | "
        f"params={n_params:.0f}M"
    )
    print(f"  ✓ Tokenizer ready: {IV2_TOKENIZER_MODEL}")

except AttributeError as e:
    print(f"  ✗ Method not found: {e}")
    if iv2_model is not None:
        print("Available likely methods:")
        print([
            m for m in dir(iv2_model)
            if "text" in m.lower()
            or "txt" in m.lower()
            or "encode" in m.lower()
            or "feat" in m.lower()
        ])
    IV2_AVAILABLE = False

except Exception as e:
    print(f"  ✗ Loading failed: {e}")
    print("  → Try: pip install decord timm einops flash-attn")
    IV2_AVAILABLE = False

Loading InternVideo2: OpenGVLab/InternVideo2_CLIP_S ...
InternVideo2Config {
  "_attn_implementation_autoset": true,
  "architectures": [
    "InternVideo2_CLIP_small"
  ],
  "auto_map": {
    "AutoConfig": "OpenGVLab/InternVideo2_CLIP_S--config.InternVideo2Config",
    "AutoModel": "OpenGVLab/InternVideo2_CLIP_S--modeling_internvideo2encoder.InternVideo2_CLIP_small"
  },
  "auto_resume": false,
  "batch_size": 64,
  "batch_size_test": 4,
  "best_key": [
    "msrvtt_1k_test_match",
    "t2v_r1"
  ],
  "compile_model": false,
  "criterion": {
    "clip_loss_ratio": [
      1.0,
      1.0
    ],
    "distill_final_features": true,
    "loss_weight": {
      "mlm": 1.0,
      "mvm": 0.0,
      "uta": 0.0,
      "vtc": 1.0,
      "vtm": 1.0
    },
    "mlm_masking_prob": 0.5,
    "vtm_hard_neg": true
  },
  "debug": false,
  "deep_fusion": false,
  "deepspeed": {
    "enable": true,
    "stage": 1
  },
  "delete_ds_optim_states": true,
  "device": "cuda",
  "dist_url": "env://",
  "evaluat

In [64]:
# ── Inspect available feature methods (run if loading raised AttributeError) ──
# This cell helps discover the correct method names for a given checkpoint variant.
if iv2_model is not None:
    feat_methods = [m for m in dir(iv2_model) if "feat" in m.lower() or "encode" in m.lower()]
    print("Available feature/encode methods on this checkpoint:")
    for m in feat_methods:
        print(f"  {m}")


Available feature/encode methods on this checkpoint:
  _prepare_encoder_decoder_kwargs_for_generation
  _tie_encoder_decoder_weights
  build_text_encoder
  build_vision_encoder
  encode_text
  encode_vision
  text_encoder
  vision_encoder


### 4. Encoding Functions

`encode_text_iv2` uses the BERT-large tokenizer and `model.get_txt_feat`.  
`encode_video_segment` samples frames uniformly from `[start_sec, end_sec]` with decord,
formats them as `(1, C, T, H, W)` (the tensor shape expected by `model.get_vid_feat`),
and returns an L2-normalised embedding.


In [65]:
import decord
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any, Tuple
import uuid

# ── Frame transform matching InternVideo2's vid2tensor preprocessing ──────────
# Source: modeling_internvideo2.py bundled in the HF repo.
_IV2_MEAN = (0.485, 0.456, 0.406)
_IV2_STD  = (0.229, 0.224, 0.225)
_iv2_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.convert("RGB") if hasattr(x, "convert") else x),
    transforms.Resize(224, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=_IV2_MEAN, std=_IV2_STD),
])


import numpy as np
import torch
import torch.nn.functional as F
from typing import List

TEXT_MAX_LENGTH = 77

def encode_text_iv2(texts: List[str]) -> np.ndarray:
    """
    Encode text with InternVideo2 text encoder.
    Returns:
        (N, IV2_DIM) float32 numpy array
    """

    if not IV2_AVAILABLE:
        raise RuntimeError("InternVideo2 not loaded.")

    toks = iv2_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LENGTH,
    ).to(_iv2_device)

    with torch.no_grad():

        if hasattr(iv2_model, "get_text_features"):
            feat = iv2_model.get_text_features(**toks)

        elif hasattr(iv2_model, "get_txt_feat"):
            try:
                feat = iv2_model.get_txt_feat(
                    toks["input_ids"],
                    toks.get("attention_mask", None),
                    toks.get("token_type_ids", None),
                )
            except TypeError:
                feat = iv2_model.get_txt_feat(
                    toks["input_ids"],
                    toks.get("attention_mask", None),
                )

        elif hasattr(iv2_model, "encode_text"):
            feat = iv2_model.encode_text(toks["input_ids"])

        else:
            raise AttributeError(
                f"No compatible text encoder found.\n"
                f"Available methods: {[m for m in dir(iv2_model) if 'text' in m.lower() or 'txt' in m.lower()]}"
            )

        feat = F.normalize(feat.float(), dim=-1)

    return feat.cpu().numpy()


def _sample_frames_pil(video_path: str, start_sec: float, end_sec: float,
                        num_frames: int) -> Optional[List]:
    """Sample `num_frames` PIL frames uniformly from [start_sec, end_sec]."""
    from PIL import Image as PILImage
    try:
        vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
        fps   = vr.get_avg_fps()
        total = len(vr)
        s = min(int(start_sec * fps), total - 1)
        e = min(int(end_sec   * fps), total - 1)
        if e <= s: e = min(s + 1, total - 1)
        idx    = np.clip(np.linspace(s, e, num_frames, dtype=int), 0, total - 1)
        frames = vr.get_batch(idx).asnumpy()   # (T, H, W, 3) uint8
        return [PILImage.fromarray(f) for f in frames]
    except Exception as ex:
        print(f"  [sample_frames] {ex}")
        return None


def encode_video_segment(
    video_path: str,
    start_sec: float,
    end_sec: float,
    num_frames: int = IV2_NUM_FRAMES,
) -> np.ndarray:

    pil_frames = _sample_frames_pil(video_path, start_sec, end_sec, num_frames)

    if pil_frames is None:
        return np.zeros(IV2_DIM, dtype=np.float32)

    frame_tensors = torch.stack(
        [_iv2_transform(f) for f in pil_frames]
    )  # (T, 3, 224, 224)

    # IMPORTANT: InternVideo2 expects (B,T,C,H,W)
    video_tensor = frame_tensors.unsqueeze(0)

    model_dtype = next(iv2_model.parameters()).dtype
    video_tensor = video_tensor.to(
        device=_iv2_device,
        dtype=model_dtype
    )

    with torch.no_grad():
        feat = iv2_model.encode_vision(video_tensor)
        feat = torch.nn.functional.normalize(
            feat.float(),
            dim=-1
        )

    return feat.cpu().numpy()[0]

In [66]:
# ── Quick sanity check ────────────────────────────────────────────────────────
# Verify that text and video embeddings are in the same space
# (cosine similarity between a relevant text and a video clip should be > 0)
if IV2_AVAILABLE:
    test_texts = [
        "attention mechanism transformer architecture",
        "a cat sleeping on a sofa",   # irrelevant control
    ]
    txt_vecs = encode_text_iv2(test_texts)
    vid_vec  = encode_video_segment(VIDEO_PATH, start_sec=0, end_sec=IV2_SEGMENT_SECS)

    sims = txt_vecs @ vid_vec   # dot product = cosine sim (L2-normalised)
    print("Text-video cosine similarity (sanity check):")
    for t, s in zip(test_texts, sims):
        print(f"  {s:+.4f}  {t!r}")
    print("\nExpected: first text more similar than control phrase.")
else:
    print("IV2 not available — skipping sanity check.")


Text-video cosine similarity (sanity check):
  +0.0811  'attention mechanism transformer architecture'
  +0.0510  'a cat sleeping on a sofa'

Expected: first text more similar than control phrase.


### 5. Video Segmentation + Qdrant Indexing

In [67]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

@dataclass
class VideoSegment:
    """Fixed-length video segment with Whisper transcript for LLM context."""
    start_sec:   float
    end_sec:     float
    transcript:  str  = ""
    source_file: str  = ""

    @property
    def timestamp_label(self) -> str:
        def fmt(s): m, sec = divmod(int(s), 60); return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        return self.transcript   # alias for metric functions

    @property
    def doc_type(self) -> str:
        return "text"

def get_transcript_for_range(segs: List[dict], start_s: float, end_s: float,
                              padding: float = 1.0) -> str:
    total = max((s["timestamp"][1] or 0) for s in segs) if segs else 0
    return " ".join(
        s["text"] for s in segs
        if (s["timestamp"][1] or total) >= (start_s - padding)
        and (s["timestamp"][0] or 0)     <= (end_s   + padding)
    ).strip()

# ── Qdrant setup ──────────────────────────────────────────────────────────────
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

if RESET_COLLECTION:
    client.delete_collection(QDRANT_IV2_ONLY)
    print(f"✓ Deleted '{QDRANT_IV2_ONLY}'")

if not client.collection_exists(QDRANT_IV2_ONLY):
    client.create_collection(
        collection_name = QDRANT_IV2_ONLY,
        vectors_config  = VectorParams(size=IV2_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created '{QDRANT_IV2_ONLY}' (dim={IV2_DIM})")
else:
    print(f"✓ Existing '{QDRANT_IV2_ONLY}' (dim={IV2_DIM})")

iv2_docstore: Dict[str, VideoSegment] = {}


✓ Existing 'iv2_only_v1' (dim=512)


In [70]:
def index_video(video_path: str, batch_size: int = IV2_BATCH_SIZE) -> List[str]:
    """
    Segment the video, encode each clip with InternVideo2 video encoder,
    attach Whisper transcript, and upsert to Qdrant.
    """
    if not IV2_AVAILABLE:
        print("InternVideo2 not available.")
        return []

    vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
    duration = len(vr) / vr.get_avg_fps()

    print(f"Video: {Path(video_path).name} | {duration:.1f}s")

    step = IV2_SEGMENT_SECS - IV2_OVERLAP_SECS
    ranges, t = [], 0.0

    while t < duration:
        e = min(t + IV2_SEGMENT_SECS, duration)
        ranges.append((t, e))

        if e >= duration:
            break

        t += step

    print(
        f"Segments: {len(ranges)} × ~{IV2_SEGMENT_SECS}s "
        f"(overlap={IV2_OVERLAP_SECS}s)"
    )

    all_ids, points = [], []

    for i in range(0, len(ranges), batch_size):
        batch = ranges[i : i + batch_size]

        for start_s, end_s in batch:
            try:
                vec = encode_video_segment(video_path, start_s, end_s)
                vec = np.asarray(vec, dtype=np.float32)

                if vec.shape[-1] != IV2_DIM:
                    raise ValueError(
                        f"Bad vector dim: got {vec.shape[-1]}, expected {IV2_DIM}"
                    )

            except Exception as e:
                print(f"  ✗ [{start_s:.0f}s → {end_s:.0f}s]: {e}")
                vec = np.zeros(IV2_DIM, dtype=np.float32)

            transcript = get_transcript_for_range(
                video_transcript_segs,
                start_s,
                end_s,
            )

            seg = VideoSegment(
                start_sec=start_s,
                end_sec=end_s,
                transcript=transcript,
                source_file=video_path,
            )

            uid = str(uuid.uuid4())
            iv2_docstore[uid] = seg
            all_ids.append(uid)

            points.append(
                PointStruct(
                    id=uid,
                    vector=vec.tolist(),
                    payload={
                        "start_sec": float(start_s),
                        "end_sec": float(end_s),
                        "timestamp": seg.timestamp_label,
                        "preview": transcript[:120],
                        "source_file": str(video_path),
                    },
                )
            )

        prog = min(i + batch_size, len(ranges))
        print(f"  [{prog}/{len(ranges)}] encoded")

    for i in range(0, len(points), 100):
        client.upsert(
            collection_name=QDRANT_IV2_ONLY,
            points=points[i : i + 100],
        )

    print(f"\n✓ Indexed {len(all_ids)} segments.")
    return all_ids


iv2_ids = index_video(VIDEO_PATH)

Video: video.mp4 | 603.1s
Segments: 101 × ~8s (overlap=2s)
  [4/101] encoded
  [8/101] encoded
  [12/101] encoded
  [16/101] encoded
  [20/101] encoded
  [24/101] encoded
  [28/101] encoded
  [32/101] encoded
  [36/101] encoded
  [40/101] encoded
  [44/101] encoded
  [48/101] encoded
  [52/101] encoded
  [56/101] encoded
  [60/101] encoded
  [64/101] encoded
  [68/101] encoded
  [72/101] encoded
  [76/101] encoded
  [80/101] encoded
  [84/101] encoded
  [88/101] encoded
  [92/101] encoded
  [96/101] encoded
  [100/101] encoded
  [101/101] encoded

✓ Indexed 101 segments.


### 6. Retrieval (text → video)

In [74]:
from sentence_transformers import CrossEncoder

if ENABLE_RERANKING:
    print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
    try:
        reranker = CrossEncoder(RERANKER_MODEL)
        RERANKER_AVAILABLE = True
        print("  ✓ Reranker ready.")
    except Exception as e:
        print(f"  ✗ {e}")
        reranker = None; RERANKER_AVAILABLE = False
else:
    reranker = None; RERANKER_AVAILABLE = False
    print("Reranking disabled.")


def iv2_retrieve(query: str) -> List[VideoSegment]:
    """
    Retrieve relevant video segments for a text query.
    1. Encode query with BERT-large (IV2 text encoder) → query vector in IV2 space.
    2. Cosine similarity search against video segment embeddings in Qdrant.
    3. Cross-encoder reranking on Whisper transcript of retrieved segments.
    """
    if not IV2_AVAILABLE:
        return []

    try:
        qvec = encode_text_iv2([query])[0].tolist()
        hits = client.search(
            collection_name=QDRANT_IV2_ONLY,
            query_vector=qvec, limit=RETRIEVER_K, with_payload=True,
        )
    except Exception as e:
        print(f"[iv2_retrieve] {e}"); return []

    candidates: List[Tuple[str, VideoSegment]] = []
    for h in hits:
        seg = iv2_docstore.get(str(h.id))
        if seg:
            candidates.append((seg.transcript[:500], seg))

    if not candidates:
        return []

    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        scores = reranker.predict([(query, t) for t, _ in candidates])
        ranked = sorted(zip(scores, [s for _, s in candidates]),
                        key=lambda x: x[0], reverse=True)
        return [s for _, s in ranked[:RERANKER_TOP_N]]
    return [s for _, s in candidates[:RERANKER_TOP_N]]


# Sanity check
test = iv2_retrieve("What is the attention mechanism in Transformers?")
print(f"\nRetrieved {len(test)} segments:")
for i, s in enumerate(test):
    print(f"  [{i}] {s.timestamp_label} | {s.transcript[:100]!r}")


Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...
  ✓ Reranker ready.

Retrieved 5 segments:
  [0] [03:54 – 04:02] | "We'll be right back. is All You Need, published by Google, solved both issues. The transformer is st"
  [1] [04:00 – 04:08] | "We'll be right back. is All You Need, published by Google, solved both issues. The transformer is st"
  [2] [03:36 – 03:44] | "We'll be right back. is All You Need, published by Google, solved both issues. The transformer is st"
  [3] [04:06 – 04:14] | "We'll be right back. is All You Need, published by Google, solved both issues. The transformer is st"
  [4] [03:00 – 03:08] | "We'll be right back. is All You Need, published by Google, solved both issues. The transformer is st"


### 7. Generation (text-only LLM)

In [ ]:
import subprocess

def get_windows_host_ip():
    try:
        ip = subprocess.check_output(
            "ip route | awk '/default/ {print $3}'",
            shell=True,
            text=True
        ).strip()
        return ip
    except Exception as e:
        raise RuntimeError(f"Could not determine Windows host IP: {e}")

WIN_HOST_IP = get_windows_host_ip()
OLLAMA_BASE_URL = f"http://{WIN_HOST_IP}:11434"

print("Windows host IP:", WIN_HOST_IP)
print("Ollama URL:", OLLAMA_BASE_URL)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_AVAILABLE = False
_gen_tok = _gen_model = _ollama_llm = None
USE_OLLAMA = False

if GENERATION_BACKEND == "ollama":
    try:
        from langchain_ollama import ChatOllama
        _ollama_llm = ChatOllama(model=GENERATION_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
        USE_OLLAMA = True; GENERATOR_AVAILABLE = True
        print(f"✓ Ollama: {GENERATION_MODEL}")
    except Exception as e:
        print(f"✗ Ollama: {e}")
else:
    print(f"Loading generator: {GENERATION_MODEL} ...")
    try:
        _gen_tok   = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
        _gen_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL, torch_dtype=HF_TORCH_DTYPE,
            device_map=HF_DEVICE_MAP, trust_remote_code=True).eval()
        GENERATOR_AVAILABLE = True
        print(f"  ✓ Generator ready.")
    except Exception as e:
        print(f"  ✗ {e}")


def generate_answer(segs: List[VideoSegment], question: str) -> str:
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"
    parts = [f"{s.timestamp_label}\n{s.transcript.strip()}"
             for s in segs if s.transcript.strip()]
    context = "\n\n".join(parts) if parts else "[No transcript context retrieved]"
    prompt = (
        "Answer the question using only the provided transcript excerpts. "
        "Each excerpt is preceded by its video timestamp. "
        "If the context is insufficient, state that explicitly.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    if USE_OLLAMA:
        return _ollama_llm.invoke(prompt).content.strip()
    messages = [{"role": "user", "content": prompt}]
    if hasattr(_gen_tok, "apply_chat_template"):
        text_in = _gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt
    inputs = _gen_tok([text_in], return_tensors="pt").to(_gen_model.device)
    with torch.no_grad():
        out = _gen_model.generate(**inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                                  do_sample=False, temperature=None,
                                  pad_token_id=_gen_tok.eos_token_id)
    return _gen_tok.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                 skip_special_tokens=True)[0].strip()


# Demo
q      = "Explain multi-head attention."
docs   = iv2_retrieve(q)
answer = generate_answer(docs, q)
print("=" * 70)
print(f"Q: {q}\n\nA: {answer}\n\nSources:")
for s in docs:
    print(f"  {s.timestamp_label} {s.transcript[:80]!r}")


✓ Ollama: mistral-nemo:latest
Q: Explain multi-head attention.

A: [04:18 – 04:26]
"Now let's unpack the architecture... Each block has two key layers an attention layer and a feed forward or mlp layer... The attention layer is where all the tokens interact while in the MLP layer each token privately refines its representation..."

Sources:
  [04:00 – 04:08] "We'll be right back. is All You Need, published by Google, solved both issues. T"
  [03:54 – 04:02] "We'll be right back. is All You Need, published by Google, solved both issues. T"
  [04:06 – 04:14] "We'll be right back. is All You Need, published by Google, solved both issues. T"
  [04:18 – 04:26] "We'll be right back. is All You Need, published by Google, solved both issues. T"
  [03:00 – 03:08] "We'll be right back. is All You Need, published by Google, solved both issues. T"


### 8. Evaluation

In [82]:
import re, statistics

TEST_SET = [
    {"question": "What is scaled dot-product attention?",
     "context_keywords": ["softmax","dot product","queries","keys","values"],
     "answer_keywords":  ["softmax","scale","queries","keys"]},
    {"question": "What is multi-head attention?",
     "context_keywords": ["multi-head","projection","parallel","heads"],
     "answer_keywords":  ["head","parallel","projection"]},
    {"question": "What optimizer was used to train the Transformer?",
     "context_keywords": ["adam","optimizer","warmup","learning rate"],
     "answer_keywords":  ["adam","warmup"]},
    {"question": "What is the role of positional encoding?",
     "context_keywords": ["positional","encoding","position","sequence"],
     "answer_keywords":  ["position","order","encoding"]},
    {"question": "How does the encoder differ from the decoder?",
     "context_keywords": ["encoder","decoder","masked","self-attention"],
     "answer_keywords":  ["encoder","decoder","masked"]},
]

STOPWORDS = {"the","a","an","is","in","of","to","and","or","it","that","this",
             "with","for","on","are","was","be","by","at","as","has","have","had","from"}

def tokenize(t): return set(re.findall(r"[a-z0-9]+", t.lower()))
def context_recall(docs, kws):
    ctx = " ".join(d.content for d in docs).lower()
    return sum(1 for k in kws if k.lower() in ctx) / max(len(kws), 1)
def answer_faithfulness(answer, docs):
    ctx = tokenize(" ".join(d.content for d in docs)) - STOPWORDS
    sents = [s.strip() for s in re.split(r"[.!?]", answer) if len(s.strip()) > 10]
    return sum(1 for s in sents if (tokenize(s) - STOPWORDS) & ctx) / max(len(sents), 1)
def answer_relevance(a, q):
    qt = tokenize(q) - STOPWORDS; at = tokenize(a) - STOPWORDS
    return len(qt & at) / max(len(qt), 1)
def retrieval_precision(docs, kws):
    if not docs or not kws: return 0.0
    return sum(1 for d in docs if any(k.lower() in d.content.lower() for k in kws)) / len(docs)
def timestamp_coverage(docs, total_dur):
    if not docs or total_dur <= 0: return 0.0
    ivs = sorted([(d.start_sec, d.end_sec) for d in docs], key=lambda x: x[0])
    merged = []
    for s, e in ivs:
        if merged and s <= merged[-1][1]: merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else: merged.append((s, e))
    return min(sum(e - s for s, e in merged) / total_dur, 1.0)

METRICS = ["context_recall","answer_faithfulness","answer_relevance",
           "retrieval_precision","timestamp_coverage"]

# get video duration for timestamp_coverage
_vr = decord.VideoReader(VIDEO_PATH, ctx=decord.cpu(0))
_video_duration = len(_vr) / _vr.get_avg_fps()

def run_eval(test_set):
    results = []
    print(f"Evaluating {len(test_set)} questions ...\n")
    for item in test_set:
        q, c_kw, a_kw = item["question"], item["context_keywords"], item["answer_keywords"]
        try:
            docs   = iv2_retrieve(q)
            answer = generate_answer(docs, q)
        except Exception as e:
            print(f"  ✗ {q[:50]}: {e}")
            results.append({"question": q, "error": str(e)}); continue
        r = {
            "question":            q,
            "context_recall":      round(context_recall(docs, c_kw), 3),
            "answer_faithfulness": round(answer_faithfulness(answer, docs), 3),
            "answer_relevance":    round(answer_relevance(answer, q), 3),
            "retrieval_precision": round(retrieval_precision(docs, a_kw), 3),
            "timestamp_coverage":  round(timestamp_coverage(docs, _video_duration), 3),
            "n_retrieved":         len(docs),
            "answer_preview":      answer[:120],
        }
        results.append(r)
        print(f"  Q: {q[:60]}")
        print(f"     recall={r['context_recall']:.2f}  faith={r['answer_faithfulness']:.2f}  "
              f"rel={r['answer_relevance']:.2f}  prec={r['retrieval_precision']:.2f}  "
              f"ts_cov={r['timestamp_coverage']:.3f}")
    return results

iv2_eval_results = run_eval(TEST_SET)


Evaluating 5 questions ...

  Q: What is scaled dot-product attention?
     recall=0.00  faith=1.00  rel=0.80  prec=0.00  ts_cov=0.056
  Q: What is multi-head attention?
     recall=0.00  faith=1.00  rel=0.75  prec=0.00  ts_cov=0.056
  Q: What optimizer was used to train the Transformer?
     recall=0.00  faith=1.00  rel=0.80  prec=0.00  ts_cov=0.060
  Q: What is the role of positional encoding?
     recall=0.25  faith=0.91  rel=0.75  prec=0.20  ts_cov=0.063
  Q: How does the encoder differ from the decoder?
     recall=0.50  faith=1.00  rel=0.80  prec=1.00  ts_cov=0.056


In [83]:
# Save results
out_path = Path(PERSIST_DIR) / "iv2_only_eval_results.json"
with open(out_path, "w") as f:
    json.dump({
        "config": {
            "iv2_model":        IV2_MODEL,
            "iv2_num_frames":   IV2_NUM_FRAMES,
            "iv2_segment_secs": IV2_SEGMENT_SECS,
            "bert_model":       BERT_MODEL,
            "whisper_model":    WHISPER_MODEL,
            "generation_model": GENERATION_MODEL,
            "video_path":       VIDEO_PATH,
            "video_duration":   _video_duration,
        },
        "results": iv2_eval_results,
    }, f, ensure_ascii=False, indent=2)

valid = [r for r in iv2_eval_results if "error" not in r]
print("=== InternVideo2-only Results ===")
print(f"  Model: {IV2_MODEL}  |  Frames/clip: {IV2_NUM_FRAMES}")
for m in METRICS:
    vals = [r[m] for r in valid]
    print(f"  {m:<24}: mean={statistics.mean(vals):.3f}  min={min(vals):.3f}  max={max(vals):.3f}")
print(f"\nSaved → {out_path}")


=== InternVideo2-only Results ===
  Model: OpenGVLab/InternVideo2_CLIP_S  |  Frames/clip: 8
  context_recall          : mean=0.150  min=0.000  max=0.500
  answer_faithfulness     : mean=0.982  min=0.909  max=1.000
  answer_relevance        : mean=0.780  min=0.750  max=0.800
  retrieval_precision     : mean=0.240  min=0.000  max=1.000
  timestamp_coverage      : mean=0.058  min=0.056  max=0.063

Saved → cache/iv2_only/iv2_only_eval_results.json


---
## 9. Cross-Strategy Comparison

| Notebook | Strategy | Source | Results file |
|---|---|---|---|
| **This notebook** | InternVideo2 video+text | MP4 | `iv2_only_eval_results.json` |

Results from `rag_pipeline_local.ipynb` (VLM summaries + PDF) and `rag_pipeline_clip.ipynb`
(CLIP shared space + PDF) are not saved to JSON by default — add the result export cell
from Section 8 of those notebooks if you want to include them here.

Missing files are skipped gracefully.


In [84]:
def load_results_safe(path: str, label: str):
    p = Path(path)
    if not p.exists():
        print(f"  ⚠  {label}: not found at {path}")
        return None
    try:
        with open(p) as f: data = json.load(f)
        return data.get("results", data) if isinstance(data, dict) else data
    except Exception as e:
        print(f"  ✗ {label}: {e}"); return None

print("Loading results ...\n")
sources = {
    "IV2 video-text (this nb)": iv2_eval_results,
    "audio (Whisper + CLAP)":   load_results_safe(RESULTS_AUDIO,  "audio"),
    "multimodal fusion":        load_results_safe(RESULTS_FUSION, "fusion"),
}

BASE = ["context_recall","answer_faithfulness","answer_relevance","retrieval_precision"]

def aggregate(results, metrics):
    valid = [r for r in (results or []) if isinstance(r, dict) and "error" not in r
                                          and all(m in r for m in metrics)]
    if not valid: return {m: float("nan") for m in metrics}
    return {m: round(statistics.mean(r[m] for r in valid), 3) for m in metrics}


Loading results ...

  ⚠  fusion: not found at ./cache/multimodal/multimodal_eval_results.json


In [85]:
# ── Aggregate comparison table ─────────────────────────────────────────────────
print("\n" + "=" * 80)
print("CROSS-STRATEGY COMPARISON")
print("=" * 80)
hdr = f"{'Strategy':<28} {'recall':>9} {'faith':>9} {'rel':>9} {'prec':>9} {'n_q':>5}"
print(hdr); print("-" * len(hdr))

best: Dict[str, Tuple[str, float]] = {}
for label, results in sources.items():
    if results is None: print(f"{label:<28}  (no data)"); continue
    valid = [r for r in results if isinstance(r, dict) and "error" not in r
                                  and all(m in r for m in BASE)]
    if not valid: print(f"{label:<28}  (no valid records)"); continue
    avgs = aggregate(results, BASE)
    print(f"{label:<28} {avgs['context_recall']:>9.3f} {avgs['answer_faithfulness']:>9.3f} "
          f"{avgs['answer_relevance']:>9.3f} {avgs['retrieval_precision']:>9.3f} {len(valid):>5d}")
    for m in BASE:
        if m not in best or avgs[m] > best[m][1]:
            best[m] = (label, avgs[m])

print("-" * len(hdr))
print("\nBest per metric:")
for m, (lbl, val) in best.items():
    print(f"  {m:<24}: {lbl}  ({val:.3f})")



CROSS-STRATEGY COMPARISON
Strategy                        recall     faith       rel      prec   n_q
--------------------------------------------------------------------------
IV2 video-text (this nb)         0.150     0.982     0.780     0.240     5
audio (Whisper + CLAP)           0.075     0.438     0.550     0.057    40
multimodal fusion             (no data)
--------------------------------------------------------------------------

Best per metric:
  context_recall          : IV2 video-text (this nb)  (0.150)
  answer_faithfulness     : IV2 video-text (this nb)  (0.982)
  answer_relevance        : IV2 video-text (this nb)  (0.780)
  retrieval_precision     : IV2 video-text (this nb)  (0.240)


In [86]:
# ── Per-question breakdown ─────────────────────────────────────────────────────
def find_q(results, question):
    if not results: return None
    return next((r for r in results if isinstance(r, dict)
                 and r.get("question") == question and "error" not in r), None)

print("\n" + "=" * 80)
print("PER-QUESTION BREAKDOWN")
print("=" * 80)

for item in TEST_SET:
    q = item["question"]
    print(f"\nQ: {q}")
    print(f"  {'Strategy':<28} {'recall':>9} {'faith':>9} {'rel':>9} {'prec':>9}")
    print(f"  {'-'*70}")
    for label, results in sources.items():
        r = find_q(results, q)
        if r is None: print(f"  {label:<28}  (no result)"); continue
        print(f"  {label:<28} {r.get('context_recall',0):>9.3f} "
              f"{r.get('answer_faithfulness',0):>9.3f} {r.get('answer_relevance',0):>9.3f} "
              f"{r.get('retrieval_precision',0):>9.3f}")

print("""
\n=== Interpretation notes ===
- Source corpora differ: IV2 operates on video transcript; audio notebooks on MP3;
  fusion on all three. Lower scores may reflect source richness, not pipeline quality.
- TEST_SET is identical → per-question metrics are directly comparable.
- timestamp_coverage is IV2/audio-specific and excluded from the joint table.
""")



PER-QUESTION BREAKDOWN

Q: What is scaled dot-product attention?
  Strategy                        recall     faith       rel      prec
  ----------------------------------------------------------------------
  IV2 video-text (this nb)         0.000     1.000     0.800     0.000
  audio (Whisper + CLAP)           0.000     1.000     0.800     0.000
  multimodal fusion             (no result)

Q: What is multi-head attention?
  Strategy                        recall     faith       rel      prec
  ----------------------------------------------------------------------
  IV2 video-text (this nb)         0.000     1.000     0.750     0.000
  audio (Whisper + CLAP)           0.000     1.000     0.750     0.000
  multimodal fusion             (no result)

Q: What optimizer was used to train the Transformer?
  Strategy                        recall     faith       rel      prec
  ----------------------------------------------------------------------
  IV2 video-text (this nb)         0.000  